In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from typing import Any, List, Tuple

class AdvanceRagImplementation:
    # init
    def __init__(self, agent: Any, reteriver: ReterivalManager):
        self.agent = agent
        self.reteriver = reteriver
        self.history = []
        
    # reterival
    def _reterive_doc(self, query: str, top_k: int, threshold: float)->list[Dict[str, Any]]:
        return self.reteriver.reteriveContext(query=query, top_k=top_k, threshold=threshold)
    
    # context with citations and source 
    def _build_context(self, reterived_doc: Any) -> Tuple[str, List[Dict]]:
        context = "".join(
            f"[{i}]\n{doc['content']}"
            for i, doc in enumerate(reterived_doc, start=1))
        
        sources = [
            {
                "citation_id": i,
                "chunk_id": doc.get("id", f"chunk_{i}"),
                "source": doc.get("metadata", {}).get(
                    "source",
                    doc.get("metadata", {}).get("source_file", "unknown"),
                ),
                "page": doc.get("metadata", {}).get("page", "unknown"),
                "score": doc.get("similarity_score"),
                "preview": doc["content"][:120] + "...",
            }
            for i, doc in enumerate(reterived_doc, start=1)
        ]
        return context, sources
    
    # Build Input (sys msg, history , question/query)
    def _build_messages(self, context: str, query: str) ->Tuple[Any, Any]:
        sys_prompt = f"""
        prompt...
        {context}
        """
        return SystemMessage(content=sys_prompt), HumanMessage(content=query)
    
    # Extract the cittaions from the answer 
    def _extract_citations(self, answer: str) -> List[int]:
        matches = re.findall(r"\[(\d+)\]|【(\d+)】", answer)
        return sorted({int(a or b) for a, b in matches})
    
    def _resolve_sources(
        self, answer: str, sources: List[Dict]
    ) -> List[Dict]:
        citation_map = {source["citation_id"]: source for source in sources}
        used_citations = self._extract_citations(answer)

        return [
            citation_map[citation_id]
            for citation_id in used_citations
            if citation_id in citation_map
        ]
        
    def query_fun(self, query: str, top_k: int = 5, threshold: float = 0.0, stream: bool = False, summarize: bool = False)-> tuple[Dict, Dict, Dict]:
        # 1. reteriver
        reterived_doc = self._reterive_doc(
            query=query, top_k=top_k, threshold=threshold
        )

        if not reterived_doc:
            print("\n[INFO] No Context Found")
            answer = "I don't have enough information in the provided documents."

            self.history.append(HumanMessage(content=query))
            self.history.append(AIMessage(content=answer))

            return {
                "question": query,
                "answer": answer,
                "sources": [],
            }

        
        # 2. build context and sources from reterived_doc
        context, sources = self._build_context(reterived_doc)
        
        # 3. set a fallback technique in case for no reterived doc 
        if not context:
            print("No Context found")
            answer = ""
            sources = []
            
        
        # 4. Buid the Message(System Message and Human Message)
        sys_message, human_message = self._build_messages(context, query)
        
        # 5. Build the execution payload (combine history + SystemMessage + query)
        input_message = [sys_message] + self.history + [human_message]
        
        # 6. Model Generation 
        if stream:
            for chunk in self.agent.stream({"messages", input_message}):
                if chunk:
                    answer += chunk.content
                    print(chunk.content, end = "", flush=True)
        else:
            resp = self.agent.invoke({"messages": input_message})
            if isinstance(resp, dict) and "messages" in resp:
                answer = resp["messages"][-1].content
            if hasattr(resp, "content"):
                answer = resp.content
            else:
                answer = str(resp)
        
        # 7. resolve used sources from the citations 
        used_sources = self._resolve_sources(answer, sources)
        
        # 8. update the history 
        self.history.append(sys_message)
        self.history.append(human_message)
        
        # 7. return eturn {'question', 'answer', 'sources', 'history'}
        return {
            "question": query,
            "answer": answer,
            "source": used_sources,
            "history": self.history
        }

summarizer = SummarizationMiddleware(
    model=llm,
    trigger=("tokens", 10000),
    keep=("messages", 4)
)
agent = create_agent(
    model=llm,
    tools=[],
    middleware=[summarizer]
)     
rag = AdvanceRagImplementation(agent=agent, reteriver=reterival)    


In [ ]:
# Reterival Pipeline
class ReterivalManager:
    def __init__(self, embedding_manager: EmbeddingManager, vector_store: VectorStore):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
    
    def reteriveContext(self, query: str, top_k: int, threshold: float = 0.0) -> list[Dict[str, Any]]:
        # Generate the qurey embedding 
        query_embedding = self.embedding_manager.generate_embedding([query])
        
        # serach in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )
            reterived_docs = []
            if results["documents"] and results["documents"][0]:
                docs = results["documents"][0]
                ids = results["ids"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                
                for i, (id_, doc, metadata, distance) in enumerate(zip(ids, docs, metadatas, distances)):
                    # convert distanbce to simlarity score (Chromadb uses cosine distance)
                    similarity_score = 1 - distance
                    if similarity_score >= threshold:
                        reterived_docs.append({
                            "id": id_,
                            "content": doc,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Reterived {len(reterived_docs)}")
            else:
                print("no document Reterived ")
            return reterived_docs
        except Exception as e:
            print(e)

reterival = ReterivalManager(embedding_manager, vectorstore_manager)